#### Tech Challenge - Fase 1 - Data Analysis and Exploration

**Contexto**

Atuando como Expert em Data Analytics em uma empresa que exporta vinhos do Brasil para o mundo todo em uma área recém-criada dentro da empresa sendo responsável pelos relatórios iniciais a serem apresentados, explicando a quantidade de vinhos exportados e os fatores externos que podem vir a surgir e que interferem nas análises:

1. Dados climáticos
2. Dados demográficos
3. Dados econômicos
4. Dados de avaliações de vinhos

Sendo necessario a criação de uma tabela contendo as seguintes informações:

a. País de origem (Brasil)
b. País de destino
c. Quantidade em litros de vinho exportado (utilize: 1KG =1L)
d. Valor em US$

Com os dados fornecido de vinícola parceira e que podem ser encontrados aqui através do seguinte link:

 http://vitibrasil.cnpuv.embrapa.br/index.php?opcao=opt_01

##### Importação de pacotes, bibliotecas e criação de funções (DEF)

In [124]:
# Importar biblioteca completa
import pandas as pd

# Importar função especifica de um módulo
from datetime import datetime

##### Tratar base de dados

In [125]:
# Caminho onde esta o arquivo
caminho_arquivo = r"C:\Users\ricar\OneDrive\Cursos\Pós Graduação\Data Analytics - FIAP\02 - Fase 1 - Data Analysis and Exploration\05 - Atividade a ser entregue até 03_06 - Em grupo\Base\ExpVinho.csv"

# Lista vazia para armazenar os dados
dados = []

with open(caminho_arquivo, encoding='utf-8') as f:
    linhas = f.readlines()

# Separar o cabeçalho original
cabecalho_raw = linhas[0].strip().split()

# Definir colunas iniciais fixas
colunas = ['Id', 'País']

# Selecionar os anos
anos = cabecalho_raw[2:]

# Agrupar pares de colunas por ano como 'Ano_Quantidade' e 'Ano_Valor'
for i in range(0, len(anos), 2):
    ano = anos[i]
    colunas.append(f"{ano}_quantidade_kg")
    colunas.append(f"{ano}_valor_usd")

# Lê os dados
for linha in linhas[1:]:
    partes = linha.strip().split()
    id = partes[0]

    # Juntar os campos do país até encontrar o início dos valores (número)
    i = 1
    while not partes[i].isdigit():
        i += 1
    pais = ' '.join(partes[1:i])
    valores = partes[i:]
    dados.append([id, pais] + valores)

# Criar o DataFrame e ajustar o nome
df_vinhos = pd.DataFrame(dados, columns=colunas)
df_vinhos = df_vinhos.rename(columns={'País': 'pais','Id':'id'})
df_vinhos.head()

,id,pais,1970_quantidade_kg,1970_valor_usd,1971_quantidade_kg,1971_valor_usd,1972_quantidade_kg,1972_valor_usd,1973_quantidade_kg,1973_valor_usd,...,2020_quantidade_kg,2020_valor_usd,2021_quantidade_kg,2021_valor_usd,2022_quantidade_kg,2022_valor_usd,2023_quantidade_kg,2023_valor_usd,2024_quantidade_kg,2024_valor_usd
0,1,Afeganistão,0,0,0,0,0,0,0,0,...,0,0,11,46,0,0,0,0,0,0
1,2,África do Sul,0,0,0,0,0,0,0,0,...,4,21,0,0,0,0,117,698,103,1783
2,3,"Alemanha, República Democrática",0,0,0,0,4168,2630,12000,8250,...,6261,32605,2698,6741,7630,45367,4806,31853,6666,48095
3,4,Angola,0,0,0,0,0,0,0,0,...,0,0,0,0,4068,4761,0,0,0,0
4,5,Anguilla,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [126]:
# Selecionar o nome das colunas
anos = df_vinhos.columns[2:]
anos

# Fazer um copia do data frame
df_vinhos_tratado = df_vinhos.copy()

# Despivotar o data frame
df_vinhos_tratado = df_vinhos.melt(id_vars=['id','pais'],value_vars=anos, var_name='ano_tipo', value_name='valor') 

# Criar duas novas coluna com o ano e o tipo
df_vinhos_tratado['tipo'] = df_vinhos_tratado['ano_tipo'].str.split('_', n=1).str[1]
df_vinhos_tratado['ano'] = df_vinhos_tratado['ano_tipo'].str.split('_', n=1).str[0]

# Remover coluna não necessaria
df_vinhos_tratado = df_vinhos_tratado.drop(columns=['ano_tipo'])

# Pivotar a coluna tipo com o valor 
df_vinhos_tratado = df_vinhos_tratado.pivot(index=['id', 'pais', 'ano'],columns='tipo',values='valor')
df_vinhos_tratado = df_vinhos_tratado.reset_index()

# Converter colunas para o formato númerico
colunas_altercao = ['quantidade_kg','valor_usd']
df_vinhos_tratado[colunas_altercao] = df_vinhos_tratado[colunas_altercao].apply(pd.to_numeric, errors='coerce')

# Filtrar ultimos dados
df_vinhos_tratado.query('pais == "África do Sul"').tail()

tipo,id,pais,ano,quantidade_kg,valor_usd
2965,2,África do Sul,2020,4,21
2966,2,África do Sul,2021,0,0
2967,2,África do Sul,2022,0,0
2968,2,África do Sul,2023,117,698
2969,2,África do Sul,2024,103,1783


##### Analise do preço médio

Pergunta = Qual a elasticidade do preço por kg com relação à quantidade exportada?

In [127]:
# Limitar os ultimos anos
delta_anos = 15
ano_atual = datetime.now().year
limite = str(ano_atual-delta_anos)

df_vinhos_ultimo = df_vinhos_tratado.copy()
df_vinhos_ultimo = df_vinhos_ultimo[df_vinhos_ultimo['ano'] >=  limite].sort_values(by='ano')
df_vinhos_ultimo


tipo,id,pais,ano,quantidade_kg,valor_usd
40,1,Afeganistão,2010,0,0
2515,14,Bahamas,2010,3175,12759
3395,27,Camarões,2010,0,0
4440,44,Cuba,2010,0,0
1250,119,São Vicente e Granadinas,2010,0,0
...,...,...,...,...,...
4399,43,Croácia,2024,23,44
2474,139,Vanuatu,2024,0,0
1264,119,São Vicente e Granadinas,2024,0,0
7259,90,Malta,2024,6302,16586


In [128]:
# Agrupar os dados
df_vinhos_ultimo_rank = df_vinhos_ultimo.copy()
df_vinhos_ultimo_rank = df_vinhos_ultimo_rank.groupby('pais')[['quantidade_kg','valor_usd']].sum().reset_index()

# Fazer o ranking
rank = 5
df_vinhos_ultimo_rank['ranking'] = df_vinhos_ultimo_rank['quantidade_kg'].rank(ascending=False, method='max')
df_vinhos_ultimo_rank = df_vinhos_ultimo_rank.query('ranking <= @rank').sort_values(by='ranking')
df_vinhos_ultimo_rank

tipo,pais,quantidade_kg,valor_usd,ranking
102,Paraguai,34021588,47591976,1.0
112,Rússia,10909283,17419774,2.0
50,Estados Unidos,3287390,9297709,3.0
65,Haiti,2797418,3906144,4.0
49,Espanha,1988248,3803901,5.0


In [ ]:
# Calcular o preço médio

df_vinhos_pm =  df_vinhos_ultimo.query('quantidade_kg > 0').copy()

# Fazer o filtro 
df_vinhos_pm = df_vinhos_pm[df_vinhos_pm['pais'].isin(df_vinhos_ultimo_rank['pais'])]

# Calcular o preço médio 
df_vinhos_pm['pm'] = (df_vinhos_pm['valor_usd'] / df_vinhos_pm['quantidade_kg']).fillna(0)

# Ordernar os dados
df_vinhos_pm = df_vinhos_pm.sort_values(by=['pais','ano'])

# Calcular a elasticidade-preço da oferta (EPO)
df_vinhos_pm['var_pct_qtd'] = df_vinhos_pm.groupby('pais')['quantidade_kg'].pct_change()
df_vinhos_pm['var_pct_pm'] = df_vinhos_pm.groupby('pais')['pm'].pct_change()

df_vinhos_pm["elasticidade"] = (df_vinhos_pm["var_pct_pm"] / df_vinhos_pm["var_pct_qtd"])

df_vinhos_pm

tipo,id,pais,ano,quantidade_kg,valor_usd,pm,var_pct_qtd,var_pct_pm,elasticidade
4936,52,Espanha,2011,5206,24618,4.728774,NaN,NaN,NaN
4938,52,Espanha,2013,1972980,3748940,1.900141,377.981944,-0.598175,-0.001583
4943,52,Espanha,2018,6123,22631,3.696064,-0.996897,0.945153,-0.948095
4944,52,Espanha,2019,3540,1353,0.382203,-0.421852,-0.896592,2.125370
4945,52,Espanha,2020,28,126,4.500000,-0.992090,10.773836,-10.859732
4948,52,Espanha,2023,180,4171,23.172222,5.428571,4.149383,0.764360
4949,52,Espanha,2024,191,2062,10.795812,0.061111,-0.534105,-8.739908
4990,53,Estados Unidos,2010,228968,478630,2.090379,NaN,NaN,NaN
4991,53,Estados Unidos,2011,306787,1030254,3.358206,0.339868,0.606506,1.784530
4992,53,Estados Unidos,2012,146585,303986,2.073787,-0.522193,-0.382472,0.732434
